In [1]:
import os
import sys
import subprocess

# 1) Quick check: nvidia-smi (works in WSL2 with NVIDIA drivers installed)
try:
    nv = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if nv.returncode == 0:
        print("nvidia-smi output:\n", nv.stdout.splitlines()[:20])
    else:
        print("nvidia-smi not available or failed (returncode={}):".format(nv.returncode))
        print(nv.stderr.strip())
except FileNotFoundError:
    print("nvidia-smi command not found. Ensure NVIDIA drivers for WSL are installed.")

# 2) Ensure CUDA device visibility (default to device 0 if not set)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# 3) PyTorch: import and configure
try:
    import torch
    torch_available = torch.cuda.is_available()
    print("PyTorch version:", torch.__version__)
    print("CUDA available (torch):", torch_available)
    if torch_available:
        print("CUDA version (torch):", torch.version.cuda)
        print("GPU count (torch):", torch.cuda.device_count())
        try:
            print("Current GPU name:", torch.cuda.get_device_name(0))
        except Exception:
            pass
        # recommended backend flags
        torch.backends.cudnn.benchmark = True
    else:
        print("Torch cannot access CUDA. Check your WSL NVIDIA driver / CUDA toolkit.")
except Exception as e:
    print("Failed to import torch:", repr(e))

# 4) TensorFlow: import and enable memory growth to avoid grabbing all GPU memory
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    gpus = tf.config.list_physical_devices("GPU")
    print("Physical GPUs (tf):", len(gpus))
    if gpus:
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except Exception:
                # fallback for older/newer tf versions
                try:
                    tf.config.set_logical_device_configuration(gpu, [])
                except Exception:
                    pass
        logical_gpus = tf.config.list_logical_devices("GPU")
        print("Logical GPUs (tf):", len(logical_gpus))
    else:
        print("TensorFlow did not detect any GPUs. Confirm TF built with CUDA support and correct drivers.")
except Exception as e:
    print("Failed to import/configure TensorFlow:", repr(e))

# 6) Summary
print("\nEnvironment summary:")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Python:", sys.version.splitlines()[0])

nvidia-smi output:
 ['Tue Nov  4 20:10:10 2025       ', '+-----------------------------------------------------------------------------------------+', '| NVIDIA-SMI 581.42                 Driver Version: 581.42         CUDA Version: 13.0     |', '+-----------------------------------------+------------------------+----------------------+', '| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |', '| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |', '|                                         |                        |               MIG M. |', '|=========================================+========================+======================|', '|   0  NVIDIA GeForce RTX 2060      WDDM  |   00000000:1F:00.0  On |                  N/A |', '| 30%   33C    P8             15W /  160W |    1304MiB /   6144MiB |     24%      Default |', '|                                         |                        |                  N/